# Deep Learning Lab 0

## Imports

In [6]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn

## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [7]:
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

test_transform = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=train_transform)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=test_transform)

Files already downloaded and verified
Files already downloaded and verified


In [8]:
trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set


In [9]:
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


cpu


In [10]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [13]:
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)

In [15]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    
    average_loss = running_loss / len(trainload)
    print(f"Epoch [{epoch+1}/{num_epochs}] Average loss: {average_loss:.3f}")

Epoch [1/10] Average loss: 1.894
Epoch [2/10] Average loss: 1.850
Epoch [3/10] Average loss: 1.820
Epoch [4/10] Average loss: 1.789
Epoch [5/10] Average loss: 1.759
Epoch [6/10] Average loss: 1.738
Epoch [7/10] Average loss: 1.719
Epoch [8/10] Average loss: 1.701
Epoch [9/10] Average loss: 1.684
Epoch [10/10] Average loss: 1.666


### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard